# MERRY: Synthetic Data Pack

Synthetic data for **MERRY**, an agentic auditor for contractor invoices and field tickets in upstream oil and gas, built for the WinLabs Snowflake hackathon.

Everything here is fictional and generated from one file, `story_bible.yaml`. The scripts turn that story into a contract, site records, and contractor invoices with planted billing errors. The answer key says exactly what MERRY should find.

---

## Folder structure

```
MERRY/                          ← your main project folder
│
├── story_bible.yaml                 ← the source of truth (edit this, then regenerate)
├── gen_contract.py                  ← script 1: makes the contract and rate card
├── gen_site_records.py              ← script 2: makes the four site-record CSVs
├── gen_invoices.py                  ← script 3: makes invoices, tickets, answer key
│
└── out/                             ← everything the scripts produce lands here
    │
    ├── contract_msa.pdf             ← from script 1
    ├── rate_card.csv                ← from script 1
    │
    ├── gate_log.csv                 ← from script 2
    ├── equipment_log.csv            ← from script 2
    ├── timesheets.csv               ← from script 2
    ├── daily_reports.csv            ← from script 2
    │
    ├── invoices/                    ← from script 3 (19 PDFs)
    │   ├── PCS-INV-1041.pdf
    │   ├── PCS-INV-1042.pdf         ← the PumpCo demo invoice
    │   └── ... (17 more)
    │
    ├── tickets/                     ← from script 3 (41 PDFs)
    │   ├── PCS-T-7101.pdf
    │   ├── LCR-T-3306.pdf           ← the unsigned ticket (replace with your handwritten photo)
    │   └── ... (39 more)
    │
    ├── handwritten_forms/           ← from script 3
    │   ├── BLANK_AQH-T-0416.pdf     ← print and fill in by hand
    │   ├── BLANK_LCR-T-3306.pdf     ← print and fill in by hand
    │   ├── BLANK_PCS-T-7111.pdf     ← print and fill in by hand
    │   └── FILL_IN_INSTRUCTIONS.txt
    │
    ├── truth_invoice_lines.csv      ← checking only, not uploaded to Snowflake
    ├── truth_tickets.csv            ← checking only
    ├── answer_key.csv               ← checking only
    └── qa_samples.csv               ← checking only
```

---

## Quick start

```bash
# 1. Install dependencies
pip install pyyaml reportlab pillow numpy
# Poppler is needed for the scanned-document effect (provides pdftoppm):
#   Mac:      brew install poppler
#   Ubuntu:   sudo apt install poppler-utils
#   Windows:  install the Poppler binaries and add them to PATH

# 2. From the MERRY/ folder, create the output folder and run the scripts in order
mkdir -p out
python gen_contract.py
python gen_site_records.py
python gen_invoices.py
```

The scripts use fixed random seeds, so every run produces identical files.

### Ground rules

- **`story_bible.yaml` is the source of truth.** To change anything (a rate, a date, a clause, an error), edit the bible and rerun the scripts. Never hand-edit files in `out/`.
- **Errors are a separate layer.** Each planted error in the bible has an `enabled` flag. Set them all to `false` and rerun to get a clean baseline that should produce zero exceptions.
- **Running `gen_invoices.py` overwrites `out/tickets/`.** Keep backup copies of your handwritten ticket photos outside `out/` and copy them back in after regenerating.

---

## The story

**Pinnacle Upstream Philippines Inc.** operates **Well Pad BK-07** in the Bukid Field. Five contractors work the site under one master service agreement (PUP-MSA-2026-014). The company representative, **Ramon Dizon**, approves every field ticket.

| Code | Contractor | Bills for |
|---|---|---|
| PCS | PumpCo Services Inc. | Pump unit PU-101, five-person crew, proppant handling |
| LCR | Liftmax Crane & Rigging Corp. | 50-ton crane CR-301, crane operator, rigger |
| AQH | AquaHaul Trucking | Water hauling, per 20 m³ load |
| KCS | Kusina Camp Services | Meals and accommodation, per head per day |
| TLW | TruLine Wireline Services | Wireline runs, engineer and operator hours |

The data covers four weeks:

| Week | Dates | What happens |
|---|---|---|
| W1 | Aug 31 – Sep 6 | Normal week. Heavy rain on **Sep 3** suspends field work. |
| W2 | Sep 7 – 13 | **Demo week.** A typhoon suspends work on **Sep 10–11**; PumpCo's pump sits idle on the pad. |
| W3 | Sep 14 – 20 | The messy week. Kusina's contract ends Sep 15; PumpCo demobilizes Sep 16. |
| W4 | Sep 21 – 27 | All clean. A baseline showing correct invoices pass. Camp closed, so no Kusina invoice. |

**The headline case:** PumpCo bills ₱500,000 for week 2, including the pump for 5 days at ₱40,000. The equipment log shows it ran only 3 days, so the rules flag ₱80,000. The agent finds the standby clause (idle equipment on location during weather is billable at half rate) and the daily reports confirming the typhoon, so it corrects the overbilling to ₱40,000 and drafts the dispute.

---

## File reference

### Source of truth and scripts

| File | What it is |
|---|---|
| `story_bible.yaml` | The full story: operator, site, weeks, weather, contract clauses, contractors and rates, what actually happened each day, and the planted errors with expected outcomes. |
| `gen_contract.py` | Reads the bible and writes the contract PDF and rate card CSV. |
| `gen_site_records.py` | Reads the bible and writes the four site-record CSVs from what actually happened. Errors never touch these files. |
| `gen_invoices.py` | Reads the bible, builds correct invoices and tickets from actual activity, applies the planted errors, renders the PDFs, and writes the truth files, answer key, and sample questions. |

### Contract (from script 1)

**`contract_msa.pdf`**: a 3-page master service agreement. Pages 1–2 contain seven articles of contract text; page 3 has Exhibit A (rates per contractor) and Exhibit B (each contractor's term dates). Clauses tied to planted errors are copied word for word from the bible, so the agent's citations match the answer key exactly. Amounts appear as "PHP" because the PDF fonts can't render ₱.

| Clause | Rule it sets | Used by |
|---|---|---|
| 2.1 Term | No payment for work outside a contractor's term | E6 |
| 4.1 Rates | Rates include fuel and consumables | E2, E7 |
| 4.2 Standby | Idle equipment on location: 50% of day rate; stood-down crew: not billable | E1a, E1b |
| 5.1 Field Tickets | Charges need a ticket signed by the company rep | E5 |
| 5.3 Hours | Hours counted gate entry to gate exit | E8 (why the rule fires) |
| 5.4 Travel Time | Travel billable, up to 2 hours each way | E8 (why it's dismissed) |
| 6.1 Invoicing | A ticket can be billed only once | E3 |
| 7.3 Exclusions | No fuel surcharges or unapproved mobilization | E7 |

**`rate_card.csv`**: the same rates as Exhibit A, in table form. 13 rows.

| Column | Meaning |
|---|---|
| `contractor_code`, `contractor_name` | Contractor |
| `rate_code` | e.g. `PCS-EQ-PU`; codes containing `-EQ-` are equipment |
| `item`, `unit` | What is billed and per what (day, hour, load, tonne, run, head-day) |
| `rate_php` | Contract rate |
| `standby_rate_php` | Half the day rate; equipment rows only |
| `term_start`, `term_end` | Contractor's contract term |

### Site records (from script 2)

These record what actually happened on site. The rules compare invoices against them.

**`gate_log.csv`** (176 rows): every entry and exit through the site gates.

| Column | Meaning |
|---|---|
| `gate_entry_id` | e.g. G-00001 |
| `entry_date` | Date of entry |
| `subject_type` | `PERSON` or `EQUIPMENT` |
| `subject_id`, `subject_name` | Worker ID (e.g. PCS-W01) or equipment ID (e.g. PU-101) |
| `contractor_code` | Contractor |
| `gate` | `MAIN` or `SECONDARY` |
| `time_in`, `time_out` | Timestamps |
| `remarks` | e.g. "Mobilized / demobilized", "Water delivery, 20 m3" |

Workers get one row per day on site. Equipment that stays on the pad (PU-101, CR-301) gets one row covering arrival to departure. Each water truck trip is its own row, so trips per day equal loads delivered.

**`equipment_log.csv`** (31 rows): daily status for the pump (PU-101), crane (CR-301), and wireline unit (WL-501).

| Column | Meaning |
|---|---|
| `log_date`, `equipment_id`, `contractor_code` | Which machine, which day |
| `status` | `OPERATING`, `IDLE` (on location, unused due to weather), `OFF_HIRE` (on location but released from paid hire), or `DEMOBILIZED` |
| `hours_run` | Engine hours that day |
| `remarks` | e.g. proppant tonnage pumped, number of wireline runs |

`OFF_HIRE` covers the weekends PU-101 sits on the pad between jobs. It separates them from weather standby, so only weather days are billable at the standby rate.

**`timesheets.csv`** (106 rows): one row per worker per day worked: `work_date`, `worker_id`, `worker_name`, `contractor_code`, `role`, `hours`, `approved_by`. TruLine's Sep 8 hours include travel (14 hours), while the gate log shows 10 on site.

**`daily_reports.csv`** (28 rows): one row per day from Aug 31 to Sep 27: `report_date`, `weather`, `ops_status` (`NORMAL` or `SUSPENDED`), `camp_headcount`, `activities`, `supervisor_notes`, `reported_by`. The supervisor notes carry the context the agent needs, such as weather holds, the pump left idle on the pad, and TruLine traveling in from its yard.

### Invoices and tickets (from script 3)

**`invoices/`** (19 PDFs, named `<contractor>-INV-<number>.pdf`): one invoice per contractor per week, with line items, ticket references, subtotal, 12% VAT, and total. All planted errors are in the line items, before VAT.

| Contractor | W1 | W2 | W3 | W4 |
|---|---|---|---|---|
| PCS | 1041 (E1a) | **1042 (E1b, demo)** | 1043 | 1044 |
| LCR | 2207 | 2208 (E2) | 2209 (E5) | 2210 |
| AQH | 318 | 319 | 320 (E3) | 321 |
| KCS | 77 | 78 | 79 (E6) | — |
| TLW | 5512 (E7) | 5513 (E8) | 5514 (E4) | 5515 |

Each contractor has its own layout (colors, fonts, date formats, column headings) to test extraction. **AquaHaul and Kusina invoices are made to look like photocopied scans** (grayscale, tilted, noisy).

**`tickets/`** (41 PDFs, named `<contractor>-T-<number>.pdf`): daily field tickets listing personnel, equipment, and quantities, each with a contractor signature and the company rep's signature. Kusina submits one weekly headcount sheet instead of daily tickets. AquaHaul and Kusina tickets are scanned-style, as are the three handwritten placeholders.

| Ticket | Note |
|---|---|
| PCS-T-7105 to 7109 | Back the PumpCo demo invoice. 7108 and 7109 are the typhoon days (pump on location, no crew). |
| AQH-T-0412 | The Sep 7 ticket billed on both the W2 and W3 AquaHaul invoices (E3). |
| LCR-T-3306 | Sep 15 ticket with no company signature (E5). Handwritten. |
| PCS-T-7111, AQH-T-0416 | Correctly signed handwritten tickets, to test extraction on handwriting. |

**`handwritten_forms/`**: blank printable versions of the three handwritten tickets, plus `FILL_IN_INSTRUCTIONS.txt` listing exactly what to write on each. Print, fill in, photograph, and save over the matching placeholder in `tickets/` with the same file name (PDF preferred). Leave the company signature blank on LCR-T-3306.

### Checking files (from script 3; never uploaded to Snowflake)

**`truth_invoice_lines.csv`**: every invoice line exactly as printed, with invoice header fields repeated on each row and a `planted_error` column. Compare AI_EXTRACT's invoice output against this.

**`truth_tickets.csv`**: every ticket's contents, plus `signed_by_company_rep`, `handwritten`, and `scanned` flags. Compare AI_EXTRACT's ticket output against this.

**`answer_key.csv`**: what MERRY should find. Columns: `error_id`, `contractor_code`, `week`, `invoice_no`, `description`, `rule`, `flagged_php`, `decision`, `correct_php`, `overbilled_php`, `withheld_php`, `clauses`, `evidence`.

**`qa_samples.csv`**: 6 Cortex Search and 7 Cortex Analyst test questions, with expected answers and sources (`tool`, `question`, `expected_answer`, `source`).

---

## Planted errors and expected results

| ID | Invoice | Error | Rule | Flagged | Agent decision | Correct amount |
|---|---|---|---|---|---|---|
| E1a | PCS-INV-1041 | Sep 3 rain day billed at full pump rate | Site-record match | ₱40,000 | Correct | ₱20,000 |
| **E1b** | **PCS-INV-1042** | **Sep 10–11 typhoon days billed at full pump rate** | **Site-record match** | **₱80,000** | **Correct** | **₱40,000** |
| E2 | LCR-INV-2208 | Rigger billed at crane operator rate, 3 days | Rate | ₱10,500 | Confirm | ₱10,500 |
| E3 | AQH-INV-320 | Ticket AQH-T-0412 billed a second time | Duplicates | ₱27,000 | Confirm | ₱27,000 |
| E4 | TLW-INV-5514 | 3 runs × ₱28,000 shown as ₱94,000 | Math | ₱10,000 | Confirm | ₱10,000 |
| E5 | LCR-INV-2209 | Sep 15 ticket not signed by company rep | Ticket backing | ₱50,500 | Confirm (withhold) | ₱50,500 |
| E6 | KCS-INV-79 | Camp billed Sep 16–20, after contract ended | Eligibility | ₱114,000 | Confirm | ₱114,000 |
| E7 | TLW-INV-5512 | Fuel surcharge billed as a separate line | Eligibility | ₱15,000 | Confirm | ₱15,000 |
| E8 | TLW-INV-5513 | 14 hours billed vs. 10 at the gate | Site-record match | ₱13,200 | **Dismiss** (travel time allowed) | ₱0 |

| Total | Amount |
|---|---|
| Flagged by the rules | ₱360,200 |
| Upheld by the agent | ₱287,000 |
| Overbilled | ₱236,500 |
| Withheld pending signature (E5) | ₱50,500 |
| Dismissed (E8) | ₱13,200 |

PumpCo's two standby errors (E1a and E1b) form the repeat pattern the dashboard highlights.

---

## Where each file goes in Snowflake

| Files | Destination | Used by |
|---|---|---|
| `contract_msa.pdf` | Document stage → AI_PARSE_DOCUMENT → clause chunks | Cortex Search (agent) |
| `invoices/*.pdf`, `tickets/*.pdf` | Document stage → AI_EXTRACT | Extracted invoice lines and tickets tables |
| `rate_card.csv` | Table | SQL rules (rate, eligibility, standby math) |
| `gate_log.csv`, `equipment_log.csv`, `timesheets.csv`, `daily_reports.csv` | Tables | SQL rules, plus the semantic view for Cortex Analyst (agent) |
| `truth_*.csv`, `answer_key.csv`, `qa_samples.csv` | **Not uploaded** | Your checks on extraction, rules, and agent output |
| `handwritten_forms/` | **Not uploaded** | Print and fill in only |

Document stages for AI functions need server-side encryption (`ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')`).